# CropHist Iowa Business Evaluation

This notebook is the buyer-facing path for the CropHist Iowa sample.

It is organized around two concrete use cases:

1. **Underwriting / diligence:** find fields with concentrated corn / soy exposure and limited recent diversity.
2. **Monoculture screening:** find fields with the longest same-crop streaks and inspect their timelines.

It also finishes with a simple 2026 baseline forecast.

The notebook uses the public sample directly from CropHist URLs. No API key is required.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display

SAMPLE_BASE = 'https://crophist.com/pages/iowa-sample/downloads'
CROP_COLORS = {
    'Corn': '#f4c430',
    'Soybeans': '#43a047',
    'Grassland/Pasture': '#7cb342',
    'Alfalfa': '#2e7d32',
    'Open_Water': '#1e88e5',
    'Developed/Open_Space': '#8d6e63',
    'No data': '#94a3b8',
    'Unknown': '#94a3b8',
}


def point_label(lon, lat):
    return f'{lon:.4f}, {lat:.4f}'


def crop_color(crop_name):
    return CROP_COLORS.get(crop_name, '#94a3b8')


def crop_chip(crop_name):
    label = crop_name if pd.notna(crop_name) and str(crop_name).strip() else 'No data'
    return (
        '<span style="white-space:nowrap;">'
        f'<span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:{crop_color(label)};margin-right:6px;vertical-align:middle;"></span>'
        f'{label}'
        '</span>'
    )


def render_crop_legend(crops):
    items = ''.join(
        f'<span style="display:inline-block;margin:0 14px 10px 0;">{crop_chip(crop)}</span>'
        for crop in crops
    )
    display(HTML(f'<div style="margin:8px 0 14px;">{items}</div>'))


def render_crop_matrix(matrix):
    html_df = matrix.fillna('No data').map(crop_chip)
    html = html_df.to_html(escape=False)
    html = html.replace('<table border="1" class="dataframe">', '<table class="crop-matrix">')
    style = '''
    <style>
      table.crop-matrix {
        border-collapse: collapse;
        font-family: Arial, sans-serif;
        font-size: 14px;
      }
      table.crop-matrix th,
      table.crop-matrix td {
        border-bottom: 1px solid #e5e7eb;
        padding: 8px 10px;
        text-align: left;
      }
      table.crop-matrix thead th {
        background: #f8fafc;
      }
      table.crop-matrix tbody tr:nth-child(even) {
        background: #f8fafc;
      }
    </style>
    '''
    display(HTML(style + html))


def baseline_forecast(last_crop):
    if last_crop == 'Corn':
        return 'Soybeans'
    if last_crop == 'Soybeans':
        return 'Corn'
    return last_crop


In [ ]:
all_history = pd.read_parquet(f'{SAMPLE_BASE}/iowa-fields-sample-long.parquet')
all_history = all_history.sort_values(['field_id', 'year']).reset_index(drop=True)
field_count = all_history['field_id'].nunique()
year_min = int(all_history['year'].min())
year_max = int(all_history['year'].max())
year_count = all_history['year'].nunique()

print(f'Loaded {field_count:,} Iowa fields across {year_count} years ({year_min}-{year_max}).')

latest_mix = (
    all_history[all_history['year'] == year_max]['crop_name']
    .value_counts()
    .rename_axis('crop_name')
    .reset_index(name='fields')
)
latest_mix['share_pct'] = (latest_mix['fields'] / latest_mix['fields'].sum() * 100).round(1)

display(latest_mix.head(12))


In [ ]:
crop_frequency = (
    all_history.groupby(['field_id', 'crop_name'])
    .size()
    .reset_index(name='years_as_top_crop')
)

crop_frequency = crop_frequency.sort_values(['field_id', 'years_as_top_crop', 'crop_name'], ascending=[True, False, True])
crop_frequency['top_crop_rank'] = crop_frequency.groupby('field_id').cumcount()
top_crop_text = (
    crop_frequency[crop_frequency['top_crop_rank'] < 4]
    .assign(item=lambda df: df['crop_name'] + ' (' + df['years_as_top_crop'].astype(int).astype(str) + ')')
    .groupby('field_id')['item']
    .agg(', '.join)
    .rename('top_crops_by_frequency')
    .reset_index()
)

recent_cutoff = year_max - 4
recent_diversity = (
    all_history[all_history['year'] >= recent_cutoff]
    .groupby('field_id')['crop_name']
    .nunique()
    .rename('recent_5y_distinct_crops')
    .reset_index()
)

streak_break = (
    all_history['field_id'].ne(all_history['field_id'].shift())
    | all_history['crop_name'].ne(all_history['crop_name'].shift())
)
all_history['streak_id'] = streak_break.cumsum()
streaks = (
    all_history.groupby(['field_id', 'streak_id'], as_index=False)
    .agg(
        streak_crop=('crop_name', 'first'),
        streak_len=('crop_name', 'size'),
    )
)
longest_streak = (
    streaks.sort_values(['field_id', 'streak_len', 'streak_crop'], ascending=[True, False, True])
    .drop_duplicates(subset=['field_id'])
    .rename(columns={'streak_crop': 'longest_streak_crop', 'streak_len': 'longest_same_crop_streak'})
    [['field_id', 'longest_streak_crop', 'longest_same_crop_streak']]
)

field_summary = (
    all_history.assign(is_corn_soy=lambda df: df['crop_name'].isin(['Corn', 'Soybeans']))
    .groupby(['field_id', 'lon', 'lat'], as_index=False)
    .agg(
        years=('year', 'nunique'),
        last_crop=('crop_name', 'last'),
        distinct_crops=('crop_name', 'nunique'),
        corn_soy_share=('is_corn_soy', 'mean'),
    )
    .merge(top_crop_text, on='field_id', how='left')
    .merge(recent_diversity, on='field_id', how='left')
    .merge(longest_streak, on='field_id', how='left')
)
field_summary['point'] = field_summary.apply(lambda row: point_label(row['lon'], row['lat']), axis=1)
field_summary['corn_soy_share_pct'] = (field_summary['corn_soy_share'] * 100).round(1)
field_summary['underwriting_flag'] = (
    (field_summary['corn_soy_share_pct'] >= 80)
    & (field_summary['recent_5y_distinct_crops'] <= 2)
)
field_summary['row_crop_monoculture_flag'] = (
    field_summary['longest_streak_crop'].isin(['Corn', 'Soybeans'])
    & (field_summary['longest_same_crop_streak'] >= 4)
)

print(f"{int(field_summary['underwriting_flag'].sum()):,} fields match the conservative underwriting concentration rule.")
print(f"{int(field_summary['row_crop_monoculture_flag'].sum()):,} fields show a 4+ year corn/soy same-crop streak.")


## Underwriting / Diligence Screen

This screen ranks fields with concentrated corn / soy exposure, limited recent diversity, and long same-crop streaks.


In [ ]:
underwriting_screen = (
    field_summary.sort_values(
        ['underwriting_flag', 'corn_soy_share_pct', 'recent_5y_distinct_crops', 'longest_same_crop_streak', 'lon', 'lat'],
        ascending=[False, False, True, False, True, True],
    )
    [
        ['lon', 'lat', 'last_crop', 'recent_5y_distinct_crops', 'corn_soy_share_pct', 'longest_streak_crop', 'longest_same_crop_streak', 'top_crops_by_frequency']
    ]
    .head(25)
)
display(underwriting_screen)


## Monoculture Screening

This screen ranks fields by same-crop streak length, surfacing corn / soy streaks first.


In [ ]:
monoculture_screen = (
    field_summary.sort_values(
        ['row_crop_monoculture_flag', 'longest_same_crop_streak', 'corn_soy_share_pct', 'lon', 'lat'],
        ascending=[False, False, False, True, True],
    )
    [
        ['lon', 'lat', 'longest_streak_crop', 'longest_same_crop_streak', 'last_crop', 'corn_soy_share_pct', 'top_crops_by_frequency']
    ]
    .head(25)
)
display(monoculture_screen)


## Representative Timelines

The sample below shows one underwriting candidate, one row-crop monoculture candidate, and one more diversified counterexample.


In [ ]:
underwriting_example = field_summary.sort_values(
    ['underwriting_flag', 'corn_soy_share_pct', 'recent_5y_distinct_crops', 'longest_same_crop_streak'],
    ascending=[False, False, True, False],
).iloc[0]
monoculture_example = field_summary[
    field_summary['field_id'] != underwriting_example['field_id']
].sort_values(
    ['row_crop_monoculture_flag', 'longest_same_crop_streak', 'corn_soy_share_pct'],
    ascending=[False, False, False],
).iloc[0]
diversified_example = field_summary[
    ~field_summary['field_id'].isin([underwriting_example['field_id'], monoculture_example['field_id']])
].sort_values(
    ['distinct_crops', 'corn_soy_share_pct'],
    ascending=[False, True],
).iloc[0]

example_points = pd.DataFrame([
    {'case': 'Underwriting candidate', **underwriting_example.to_dict()},
    {'case': 'Monoculture candidate', **monoculture_example.to_dict()},
    {'case': 'Diversified counterexample', **diversified_example.to_dict()},
]).drop_duplicates(subset=['field_id']).copy()
example_points['point'] = example_points.apply(lambda row: point_label(row['lon'], row['lat']), axis=1)

example_history = all_history[all_history['field_id'].isin(example_points['field_id'])].copy()
example_history = example_history.merge(example_points[['field_id', 'point', 'case']], on='field_id', how='left')
crop_matrix = example_history.pivot(index='year', columns='point', values='crop_name').sort_index()
present_crops = sorted(example_history['crop_name'].dropna().unique())

display(example_points[['case', 'lon', 'lat', 'last_crop', 'top_crops_by_frequency']])
render_crop_legend(present_crops)
render_crop_matrix(crop_matrix)


In [ ]:
ordered_points = example_points['point'].tolist()
fig, ax = plt.subplots(figsize=(12, 1.8 + 0.9 * len(ordered_points)))

for idx, point in enumerate(ordered_points):
    subset = example_history[example_history['point'] == point].sort_values('year')
    ax.scatter(
        subset['year'],
        [idx] * len(subset),
        s=220,
        c=subset['crop_name'].map(crop_color),
        edgecolors='#111827',
        linewidths=1.0,
    )

ax.set_title('Representative 18-year rotation timelines')
ax.set_xlabel('Year')
ax.set_xticks(sorted(example_history['year'].unique()))
ax.set_yticks(range(len(ordered_points)))
ax.set_yticklabels(ordered_points)
ax.grid(axis='x', alpha=0.25)
for spine in ['right', 'top']:
    ax.spines[spine].set_visible(False)
plt.show()


## Simple 2026 Baseline Forecast

This final step applies a naive corn/soy toggle baseline and leaves other last crops unchanged.


In [ ]:
forecast_table = field_summary.copy()
forecast_table['forecast_2026'] = forecast_table['last_crop'].map(baseline_forecast)

forecast_mix = (
    forecast_table['forecast_2026']
    .value_counts()
    .rename_axis('forecast_2026')
    .reset_index(name='fields')
)
forecast_mix['share_pct'] = (forecast_mix['fields'] / forecast_mix['fields'].sum() * 100).round(1)
display(forecast_mix.head(12))

forecast_screen = (
    forecast_table.sort_values(['corn_soy_share_pct', 'lon', 'lat'], ascending=[False, True, True])
    [
        ['lon', 'lat', 'last_crop', 'forecast_2026', 'corn_soy_share_pct', 'top_crops_by_frequency']
    ]
    .head(25)
)
display(forecast_screen)
